In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, average_precision_score

In [39]:
df = pd.read_csv('creditcard.csv')

In [43]:
n_estimators_options = [50, 100, 300, 500]
max_depth_options    = [5, 10, 20, None]

In [10]:
y  = df['Class']
df.drop(columns="Class", inplace=True)
X_all = df

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=42
)

results = []

for n_estimators in n_estimators_options:
    for max_depth in max_depth_options:
        print(f"Training n_estimators={n_estimators}, max_depth={max_depth}...")
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            class_weight='balanced',
            random_state=42,
            min_samples_leaf=15
        )

        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            'n_estimators': n_estimators,
            'max_depth':    max_depth,
            'recall':       recall_score(y_test, y_pred),
            'precision':    precision_score(y_test, y_pred),
            'f1':           f1_score(y_test, y_pred),
            'pr_auc':       average_precision_score(y_test, y_proba),
        })

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
print(results_df.to_string(index=False))
results_df.to_csv('new_grid_search_results.csv', index=False)

Training n_estimators=50, max_depth=5...
Training n_estimators=50, max_depth=10...
Training n_estimators=50, max_depth=20...
Training n_estimators=50, max_depth=None...
Training n_estimators=100, max_depth=5...
Training n_estimators=100, max_depth=10...
Training n_estimators=100, max_depth=20...
Training n_estimators=100, max_depth=None...
Training n_estimators=300, max_depth=5...
Training n_estimators=300, max_depth=10...
Training n_estimators=300, max_depth=20...
Training n_estimators=300, max_depth=None...
Training n_estimators=500, max_depth=5...
Training n_estimators=500, max_depth=10...
Training n_estimators=500, max_depth=20...
Training n_estimators=500, max_depth=None...
 n_estimators  max_depth   recall  precision       f1   pr_auc
           50        NaN 0.816327   0.792079 0.804020 0.826743
           50       20.0 0.826531   0.794118 0.810000 0.826534
           50       10.0 0.857143   0.743363 0.796209 0.814137
          300       20.0 0.836735   0.759259 0.796117 0.8090

In [29]:
new_results = pd.read_csv('new_grid_search_results.csv')

In [42]:
metrics = ['recall', 'precision', 'f1', 'pr_auc']

for metric in metrics:
    best = new_results.sort_values(metric, ascending=False).iloc[0]
    
    print(f"\n══════════════════════════════════════════════════════")
    print(f"BEST MODEL BY {metric.upper()}")
    print(f"══════════════════════════════════════════════════════")
    print(f"  Recall:       {best['recall']:.4f}")
    print(f"  Precision:    {best['precision']:.4f}")
    print(f"  F1:           {best['f1']:.4f}")
    print(f"  PR-AUC:       {best['pr_auc']:.4f}")
    print(f"  n_estimators: {best['n_estimators']}")
    print(f"  max_depth:    {best['max_depth']}")


══════════════════════════════════════════════════════
BEST MODEL BY RECALL
══════════════════════════════════════════════════════
  Recall:       0.8878
  Precision:    0.3283
  F1:           0.4793
  PR-AUC:       0.7436
  n_estimators: 50.0
  max_depth:    5.0

══════════════════════════════════════════════════════
BEST MODEL BY PRECISION
══════════════════════════════════════════════════════
  Recall:       0.8265
  Precision:    0.7941
  F1:           0.8100
  PR-AUC:       0.8265
  n_estimators: 50.0
  max_depth:    20.0

══════════════════════════════════════════════════════
BEST MODEL BY F1
══════════════════════════════════════════════════════
  Recall:       0.8265
  Precision:    0.7941
  F1:           0.8100
  PR-AUC:       0.8265
  n_estimators: 50.0
  max_depth:    20.0

══════════════════════════════════════════════════════
BEST MODEL BY PR_AUC
══════════════════════════════════════════════════════
  Recall:       0.8163
  Precision:    0.7921
  F1:           0.8040
  P